Product Name: MULTI A3(Ask-Analyze-Acute)

Requirements

In [1]:
!pip install langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers gradio pandas openai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are ins

In [2]:
!pip install rank_bm25

Importing the Libraries

In [3]:
import os
import pandas as pd
import gradio as gr

from google.colab import userdata
from openai import OpenAI

from langchain_core.documents import Document   # ✅ FIXED
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever

bm25_retriever = None

Setting up the LLM

In [4]:
# 🔑 Get key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("StepUp")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1"
)

def call_llm(prompt):
    response = client.chat.completions.create(
        model="stepfun/step-3.5-flash:free",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

PDFEater Function- Injesting pdf's and Releasing Chunks

In [5]:
def PDFEater(pdf_file):

    loader = PyPDFLoader(pdf_file.name)  # ✅ REAL FIX
    pages = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(pages)

    for c in chunks:
        c.metadata["source_type"] = "PDF"
        c.metadata["source_name"] = pdf_file.name

    return chunks

CSVEater- Similar logic as PDFEater but now for CSV(Comma Seperated Values) files

In [6]:
def CSVEater(csv_file):

    import pandas as pd
    from langchain_core.documents import Document

    df = pd.read_csv(csv_file.name)   # ✅ FIX

    docs = []

    for i, row in df.iterrows():
        text = ", ".join([f"{col}: {row[col]}" for col in df.columns])

        docs.append(Document(
            page_content=text,
            metadata={
                "source_type": "CSV",
                "source_name": csv_file.name,
                "row": i
            }
        ))

    return docs

MARKDOWNEater- Similar logic as above 2 "Eaters" but for MARKDOWN(Unstructured Data)

In [7]:
def MARKDOWNEater(md_file):

    from langchain_core.documents import Document

    with open(md_file.name, "r", encoding="utf-8") as f:
        text = f.read()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_text(text)

    docs = []

    for chunk in chunks:
        docs.append(Document(
            page_content=chunk,
            metadata={
                "source_type": "Markdown",
                "source_name": md_file.name
            }
        ))

    return docs

In [8]:
def MYMULTIWIKI(all_chunks):

    global vectorstore, bm25_retriever

    if not all_chunks:
        return "❌ No valid content found in uploaded files."

    # Dense retrieval
    vectorstore = FAISS.from_documents(all_chunks, embedding_model)

    # BM25 retrieval
    bm25_retriever = BM25Retriever.from_documents(all_chunks)

    return "📚 MultiWIKI Ready with Hybrid Retrieval!"

Knowledge Base- MYMULTIWIKI for all the 3 types of sources storage(using FAISS Vector Database)

In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = None

def MYMULTIWIKI(all_chunks):

    global vectorstore, bm25_retriever

    # Dense retrieval
    vectorstore = FAISS.from_documents(all_chunks, embedding_model)

    # BM25 keyword retrieval
    bm25_retriever = BM25Retriever.from_documents(all_chunks)

    return "📚 MultiWIKI Ready with Hybrid Retrieval!"

/tmp/ipykernel_634/1213429841.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

V.2.0 Update!!!!!- Adding Hybrid(Dense+Keyword) Retrieval Functionality(V1.0 offered Only Dense Retrieval) using BM25

In [10]:
def hybrid_retrieve(query, k=4):

    # Dense retrieval
    dense_docs = vectorstore.similarity_search(query, k=k)

    # BM25 retrieval
    keyword_docs = bm25_retriever.invoke(query)

    # Merge + remove duplicates
    combined = list({doc.page_content: doc for doc in dense_docs + keyword_docs}.values())

    return combined[:k]

AskMe()- for General Multi Source Q&A

In [11]:
def AskMe(query):

    docs = hybrid_retrieve(query, k=4)  # V2.0 Hybrid Retrieval

    context = "\n\n".join([
        f"[{doc.metadata.get('source_name')}]\n{doc.page_content}"
        for doc in docs
    ])

    prompt = f"""
Answer the question using ONLY the context below.
Cite sources clearly.

Context:
{context}

Question:
{query}
"""

    answer = call_llm(prompt)

    # 🔥 Correct citations logic
    citations_dict = {}

    for i, doc in enumerate(docs):

        source = doc.metadata.get("source_name")

        if i == 0:
            conf = "high"
        elif i <= 2:
            conf = "medium"
        else:
            conf = "low"

        if source not in citations_dict:
            citations_dict[source] = conf
        else:
            if conf == "high":
                citations_dict[source] = "high"
            elif conf == "medium" and citations_dict[source] == "low":
                citations_dict[source] = "medium"

    citations = "\n".join([
        f"{src} (confidence: {conf})"
        for src, conf in citations_dict.items()
    ])

    return f"{answer}\n\nSources:\n{citations}"

Analyzer()- Analyze the contents of documents(MVP Version,new upgrade once this works)

In [12]:
def Analyzer(query):

    # 🔹 Hybrid retrieval (Dense + BM25)
    docs = hybrid_retrieve(query, k=6)

    context = "\n\n".join([
        f"[{doc.metadata.get('source_name')}]\n{doc.page_content}"
        for doc in docs
    ])

    prompt = f"""
You are a research assistant that compares information across multiple sources.

The sources may include:
- PDF documentation
- CSV datasets
- Markdown notes

Instructions:
1. Analyze the context carefully.
2. If CSV data is present, validate claims using the dataset values.
3. Compare statements between sources.
4. Identify agreements, contradictions, or missing data.
5. Clearly explain your reasoning.

Context:
{context}

User Query:
{query}
"""

    answer = call_llm(prompt)

    # 🔹 Citation logic (rank-based confidence)
    citations_dict = {}

    for i, doc in enumerate(docs):

        source = doc.metadata.get("source_name")

        if i == 0:
            conf = "high"
        elif i <= 2:
            conf = "medium"
        else:
            conf = "low"

        if source not in citations_dict:
            citations_dict[source] = conf
        else:
            if conf == "high":
                citations_dict[source] = "high"
            elif conf == "medium" and citations_dict[source] == "low":
                citations_dict[source] = "medium"

    citations = "\n".join([
        f"{src} (confidence: {conf})"
        for src, conf in citations_dict.items()
    ])

    return f"{answer}\n\nSources:\n{citations}"

TellInShort()- Summarizer

In [13]:
def TellInShort(query):

    docs_and_scores = vectorstore.similarity_search_with_score(query, k=8)

    context = "\n\n".join([
        doc.page_content for doc, score in docs_and_scores
    ])

    prompt = f"""
Provide a clear and structured summary based on the context below.

Context:
{context}
"""

    return call_llm(prompt)

Hero(): Our Agent Dispatcher Based on Intent Classification of the User Query

In [18]:
def Hero(query):

    classifier_prompt = f"""
Classify the user query into ONE category:

1. ask → direct factual question
2. analyze → comparison or correlation across sources
3. summarize → overview or summary request

Query:
{query}

Return ONLY one word.
"""

    intent = call_llm(classifier_prompt).strip().lower()

    if "analyze" in intent:
        result = Analyzer(query)
        tool_used = "Analyze"

    elif "summarize" in intent:
        result = TellInShort(query)
        tool_used = "Acute"

    else:
        result = AskMe(query)
        tool_used = "Ask"

    return f"🧠 Tool Used: {tool_used}\n\n{result}"

Just for testing functionalities of RAG without the UI, we can use the below code:

Now Time to Add the UI Using Gradio

In [26]:
def process_files(pdf_files, csv_files, md_files):

    global vectorstore

    try:

        all_chunks = []

        if pdf_files:
            for f in pdf_files:
                print("Processing PDF:", f.name)
                chunks = PDFEater(f)
                print("PDF chunks:", len(chunks))
                all_chunks.extend(chunks)

        if csv_files:
            for f in csv_files:
                print("Processing CSV:", f.name)
                chunks = CSVEater(f)
                print("CSV chunks:", len(chunks))
                all_chunks.extend(chunks)

        if md_files:
            for f in md_files:
                print("Processing MD:", f.name)
                chunks = MARKDOWNEater(f)
                print("MD chunks:", len(chunks))
                all_chunks.extend(chunks)

        print("TOTAL CHUNKS:", len(all_chunks))

        if len(all_chunks) == 0:
            return "❌ No content loaded."

        return MYMULTIWIKI(all_chunks)

    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(error_msg)   # shows in console
        return f"❌ ERROR:\n{error_msg}"   # shows in UI
def ask_query(query):

    if vectorstore is None:
        return "⚠️ Please upload files first."

    return Hero(query)


def UI():

    with gr.Blocks() as demo:

        gr.Markdown("# 🔎 Multi-A3 (Ask–Analyze–Acute)")

        gr.Markdown("""
###  Use Cases

**1. CSV vs Documentation Validation**
Compare structured data (CSV) with technical documentation (PDF).

**2. GitHub Project Q&A**
Ask questions using README (Markdown), dataset (CSV), and documentation (PDF).
""")

        # 🔥 Upload Section (3 Columns)
        with gr.Row():

            with gr.Column():
                pdf_input = gr.File(file_count="multiple", file_types=[".pdf"], label="Upload PDF")

            with gr.Column():
                csv_input = gr.File(file_count="multiple", file_types=[".csv"], label="Upload CSV")

            with gr.Column():
                md_input = gr.File(file_count="multiple", file_types=[".md"], label="Upload Markdown")

        upload_btn = gr.Button("Process Files")
        upload_output = gr.Textbox(label="MultiWiki Status:")

        # 🔥 Question Section
        gr.Markdown("# Now that you have provided the files to MultiA3, How about you have a Conversation with him?")
        query_input = gr.Textbox(label="Chat with MultiA3: ",placeholder="Example: Is the sales report consistent with the CSV data?")
        query_btn = gr.Button("Ask")
        answer_output = gr.Textbox(lines=20)

        upload_btn.click(process_files, [pdf_input, csv_input, md_input], upload_output)
        query_btn.click(ask_query, query_input, answer_output)

    demo.launch()

Final Testing of our Application Along with UI

Testing V2.0

In [27]:
UI()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f0c90bfc232a7b9005.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
